In [1]:
import os
print(os.path.exists("model/dangerous.onnx"))

False


In [2]:
import tensorrt as trt

TRT_LOGGER = trt.Logger(trt.Logger.INFO)
builder = trt.Builder(TRT_LOGGER)

# 兼容新旧版本创建网络
if hasattr(trt.NetworkDefinitionCreationFlag, "EXPLICIT_BATCH"):
    network = builder.create_network(1 << int(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH))
else:
    network = builder.create_network(0)

parser = trt.OnnxParser(network, TRT_LOGGER)

with open("models/dangerous.onnx", "rb") as f:
    if not parser.parse(f.read()):
        for i in range(parser.num_errors):
            print(parser.get_error(i))
        raise RuntimeError("ONNX解析失败")

config = builder.create_builder_config()
# 兼容新旧版本设置工作空间
if hasattr(config, "max_workspace_size"):
    config.max_workspace_size = 8 << 30
else:
    config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 8 << 30)

# 新版TensorRT兼容的FP16开启方式
try:
    config.set_flag(trt.BuilderFlag.FP16)
    print("成功启用FP16量化加速")
except Exception as e:
    print(f"FP16加速不可用，将使用FP32精度，异常信息：{e}")

serialized_engine = builder.build_serialized_network(network, config)
with open("models/dangerous.engine", "wb") as f:
    f.write(serialized_engine)
print("ONNX转TensorRT引擎完成！")

FileNotFoundError: [Errno 2] No such file or directory: 'models/dangerous.onnx'